# Analyse: Schnittdynamik


In [1]:
import pandas as pd
from tqdm import tqdm
import numpy as np
from pathlib import Path
import cv2


In [2]:
PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = Path('../../data')
MODEL_DIR = PROJECT_ROOT / 'models' / '03_visual_cuts'

COMBINED_CSV = DATA_DIR / '03_datasets' / 'influencer_balanced' / '01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv'
OUTPUT_CSV = DATA_DIR / '04_analysis_results' / 'visual_features' / '03_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_cuts.csv'

# Kombinierten MP4-Ordner bevorzugen; sonst quellenspezifische Ordner nutzen
VIDEO_ROOT = DATA_DIR / '02_media/stratified_sample/videos'
VIDEO_ROOTS_FALLBACK = {
    'ai': DATA_DIR / '02_media/ai_videos_2025/videos',
    'real': DATA_DIR / '02_media/real_videos_2025/videos',
}

SOURCE_FILTER = ['ai', 'real']
DIFF_THRESHOLD = 25.0
MIN_GAP_BETWEEN_CUTS = 1

print(f'Input CSV: {COMBINED_CSV}')
print(f'Output CSV: {OUTPUT_CSV}')
print(f'Primary video folder: {VIDEO_ROOT}')


Input CSV: ..\..\data\03_datasets\influencer_balanced\01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv
Output CSV: ..\..\data\04_analysis_results\visual_features\03_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_cuts.csv
Primary video folder: ..\..\data\02_media/stratified_sample/videos


In [5]:
df = pd.read_csv(COMBINED_CSV)
if SOURCE_FILTER:
    df = df[df['influencer_type'].isin(SOURCE_FILTER)].copy()

def resolve_video_path(video_id: str, source: str) -> Path | None:
    primary = VIDEO_ROOT / f'{video_id}.mp4'
    if primary.is_file():
        return primary

    fallback_root = VIDEO_ROOTS_FALLBACK.get(source)
    if fallback_root is not None:
        fallback = fallback_root / f'{video_id}.mp4'
        if fallback.is_file():
            return fallback

    return None

def has_video(video_id: str, source: str) -> bool:
    return resolve_video_path(video_id, source) is not None

df['has_video'] = [has_video(str(row['video_id']), row['influencer_type']) for _, row in df.iterrows()]
df = df[df['has_video']].reset_index(drop=True)
print(f'Restricted to {len(df)} rows with available videos')
df[['video_id', 'influencer_type', 'has_video']].head()


Restricted to 500 rows with available videos


,video_id,influencer_type,has_video
0,7516243181650988334,ai,True
1,7515925552549678378,ai,True
2,7521159314757684535,ai,True
3,7521486299098778894,ai,True
4,7521490969468865847,ai,True


In [6]:
def detect_cuts_from_video(
    video_path: Path,
    threshold: float = DIFF_THRESHOLD,
    min_gap: int = MIN_GAP_BETWEEN_CUTS,
) -> tuple[int, int, float]:
    if not video_path.exists():
        return 0, 0, float('nan')

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return 0, 0, float('nan')

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    if fps <= 0:
        fps = float('nan')

    cuts = 0
    frames_scanned = 0
    prev_gray = None
    frames_since_last_cut = min_gap

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frames_scanned += 1

        if prev_gray is not None:
            diff = cv2.absdiff(gray, prev_gray)
            score = float(np.mean(diff))
            if score > threshold and frames_since_last_cut >= min_gap:
                cuts += 1
                frames_since_last_cut = 0
            else:
                frames_since_last_cut += 1

        prev_gray = gray

    cap.release()
    return cuts, frames_scanned, fps


In [8]:
# Erkennung für alle Videos ausführen
cut_counts = []
frames_used = []
fps_values = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    vid_id = str(row['video_id'])
    source = row['influencer_type']
    video_path = resolve_video_path(vid_id, source)

    if video_path is None:
        cut_counts.append(float('nan'))
        frames_used.append(0)
        fps_values.append(float('nan'))
        continue

    cuts, total_frames, fps = detect_cuts_from_video(video_path)
    cut_counts.append(cuts)
    frames_used.append(total_frames)
    fps_values.append(fps)

df['cut_count'] = cut_counts
df['frames_scanned'] = frames_used
df['video_fps'] = fps_values
df['cuts_per_frame'] = df['cut_count'] / df['frames_scanned'].replace(0, np.nan)
df['video_duration_seconds_est'] = df['frames_scanned'] / df['video_fps'].replace(0, np.nan)
df['cuts_per_second'] = df['cut_count'] / df['video_duration_seconds_est'].replace(0, np.nan)

df.to_csv(OUTPUT_CSV, index=False)
print(f'? Saved cut metrics to {OUTPUT_CSV}')


100%|██████████| 500/500 [17:13<00:00,  2.07s/it]

? Saved cut metrics to ..\..\data\04_analysis_results\visual_features\03_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_cuts.csv


In [10]:
df[['influencer_type', 'cut_count', 'cuts_per_frame', 'cuts_per_second']].groupby('influencer_type').agg(['count', 'mean', 'std']).round(3)


cut_count                 cuts_per_frame                \
                    count    mean     std          count   mean    std   
influencer_type                                                          
ai                    250   3.900  10.101            250  0.006  0.017   
real                  250  14.952  31.970            250  0.018  0.033   

                cuts_per_second                
                          count   mean    std  
influencer_type                                
ai                          250  0.182  0.485  
real                        250  0.524  0.979